In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import faiss
import json

In [2]:
DATA_DIR = "../Datasets/archive/Teeth_DataSet/Teeth_Dataset"
TRAINING_DIR = os.path.join(DATA_DIR, "Training")
VALIDATION_DIR = os.path.join(DATA_DIR, "Validation")

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.001

In [4]:
CLASSES = ['CaS', 'CoS', 'Gum', 'MC', 'OC', 'OLP', 'OT']
NUM_CLASSES = len(CLASSES)

In [5]:
MODEL_SAVE_PATH = "teeth_case_based_model.h5"
FAISS_INDEX_PATH = "oral_cases.index"
EMBEDDINGS_PATH = "case_embeddings.npy"
LABELS_PATH = "case_labels.npy"
IDS_PATH = "case_ids.json"

In [6]:
def create_data_generators():
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        zoom_range=0.2,
        shear_range=0.2,
        fill_mode='nearest'
    )

    val_datagen = ImageDataGenerator(rescale=1./255)

    return train_datagen, val_datagen

In [7]:
train_datagen, val_datagen = create_data_generators()

train_generator = train_datagen.flow_from_directory(
    TRAINING_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASSES,
    shuffle=True
)

Found 3088 images belonging to 7 classes.


In [8]:
validation_generator = val_datagen.flow_from_directory(
    VALIDATION_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False
)

Found 1028 images belonging to 7 classes.


In [9]:
def build_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Block 2
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Block 3
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Block 4
    x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Feature vector
    x = layers.GlobalAveragePooling2D()(x)

    # Embedding Layer
    embedding = layers.Dense(256, activation="relu", name="embedding")(x)
    embedding = layers.BatchNormalization()(embedding)

    # Classification Head
    x = layers.Dropout(0.5)(embedding)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation="softmax", name="classification")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model


model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,277,223 (4.87 MB)

 Trainable params: 1,274,535 (4.86 MB)

 Non-trainable params: 2,688 (10.50 KB)

In [10]:
optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [11]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7)
]

In [12]:
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks
)

Epoch 1/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 426s 4s/step - accuracy: 0.1911 - loss: 2.6997 - val_accuracy: 0.1449 - val_loss: 4.7581 - learning_rate: 0.0010
Epoch 2/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 415s 4s/step - accuracy: 0.2396 - loss: 2.2323 - val_accuracy: 0.1449 - val_loss: 8.8142 - learning_rate: 0.0010
Epoch 3/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 412s 4s/step - accuracy: 0.2775 - loss: 2.0670 - val_accuracy: 0.1994 - val_loss: 2.8770 - learning_rate: 0.0010
Epoch 4/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 408s 4s/step - accuracy: 0.2902 - loss: 1.9004 - val_accuracy: 0.1449 - val_loss: 7.0477 - learning_rate: 0.0010
Epoch 5/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 409s 4s/step - accuracy: 0.3093 - loss: 1.8168 - val_accuracy: 0.2374 - val_loss: 2.2875 - learning_rate: 0.0010
Epoch 6/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 408s 4s/step - accuracy: 0.3378 - loss: 1.7066 - val_accuracy: 0.3609 - val_loss: 1.6375 - learning_rate: 0.0010
Epoch 7/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 409s 4s/step - accuracy: 0.3572 - loss: 1.6565 - 

In [13]:
model.save(MODEL_SAVE_PATH)
print(f"Model saved as {MODEL_SAVE_PATH}")

Model saved as teeth_case_based_model.h5


In [14]:
embedding_model = keras.Model(
    inputs=model.input,
    outputs=model.get_layer("embedding").output
)

In [15]:
def build_case_database(generator, embedding_model):
    generator.reset()

    embeddings_list = []
    labels_list = []
    filenames_list = []

    for i in range(len(generator)):
        batch_imgs, batch_labels = generator[i]

        batch_embeddings = embedding_model.predict(batch_imgs, verbose=0)

        embeddings_list.append(batch_embeddings)
        labels_list.append(np.argmax(batch_labels, axis=1))

        batch_filenames = generator.filenames[i*generator.batch_size:(i+1)*generator.batch_size]
        filenames_list.extend(batch_filenames)

    embeddings_array = np.vstack(embeddings_list).astype("float32")
    labels_array = np.hstack(labels_list)

    return embeddings_array, labels_array, filenames_list

In [16]:
print("\nBuilding case embeddings from training set...")
case_embeddings, case_labels, case_ids = build_case_database(train_generator, embedding_model)

print("Embeddings ready ")
print("Total cases:", len(case_ids))


Building case embeddings from training set...
Embeddings ready 
Total cases: 3088


In [17]:
print("\nBuilding FAISS index...")
faiss.normalize_L2(case_embeddings)

embedding_dim = case_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(case_embeddings)

print("FAISS Index Ready ")
print("Total indexed:", index.ntotal)


Building FAISS index...
FAISS Index Ready 
Total indexed: 3088


In [18]:
np.save(EMBEDDINGS_PATH, case_embeddings)
np.save(LABELS_PATH, case_labels)

with open(IDS_PATH, "w") as f:
    json.dump(case_ids, f, indent=4)

faiss.write_index(index, FAISS_INDEX_PATH)